# MatchFormer Epipolar Fine-tuning — Google Colab

Fine-tunes MatchFormer Lite-LA with epipolar supervision on ScanNet.
Checkpoints are saved to **Google Drive** so training survives Colab disconnects.

**Run order (fresh session):** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7  
**Resume after disconnect:** Cell 1 → 3 → 4 → 7 (skip 2, 5, 6)

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ZIPS    = '/content/drive/MyDrive/final_proj/scannet_zips'
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/matchformer_checkpoints_run2'

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Zips dir:       {DRIVE_ZIPS}')
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zips dir:       /content/drive/MyDrive/final_proj/scannet_zips
Checkpoint dir: /content/drive/MyDrive/final_proj/matchformer_checkpoints_run2


## Cell 2 — Clone Repo & Install Dependencies
*(Skip on resume — repo and packages persist for the session)*

In [2]:
REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/final_proj/MatchFormer

!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Done.')

Repo already cloned — pulling latest
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 23 (delta 11), reused 23 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (23/23), 11.66 MiB | 11.74 MiB/s, done.
From https://github.com/sid-raj-uc/match-former
   0ecabc7..be05762  main       -> origin/main
Updating 0ecabc7..be05762
Fast-forward
 MatchFormer/colab_finetune.ipynb              | 1109 +------------------------
 MatchFormer/colab_finetune_new.ipynb          |  415 +++++++++
 MatchFormer/model/backbone/coarse_matching.py |    5 -
 MatchFormer/model/data.py                     |    5 -
 MatchFormer/model/lightning_loftr.py          |    1 -
 MatchFormer/model/losses.py                   |   13 -
 MatchFormer/model/supervision.py              |   13 -
 MatchFormer/model/utils/comm.py               |    2 -
 MatchFormer/model/utils/metrics.py            |    1 -
 MatchFormer/reports/

## Cell 3 — Verify Zips on Drive

In [5]:
import glob, os

%cd /content/final_proj/MatchFormer

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) on Drive:')
for z in zips:
    size_gb = os.path.getsize(z) / 1e9
    print(f'  {os.path.basename(z)}  ({size_gb:.2f} GB)')

/content/final_proj/MatchFormer
Found 11 zip(s) on Drive:
  scene0000_00.zip  (2.24 GB)
  scene0001_00.zip  (0.58 GB)
  scene0002_00.zip  (2.22 GB)
  scene0003_00.zip  (0.60 GB)
  scene0004_00.zip  (0.28 GB)
  scene0005_00.zip  (0.36 GB)
  scene0006_00.zip  (0.77 GB)
  scene0007_00.zip  (0.40 GB)
  scene0008_00.zip  (0.39 GB)
  scene0009_00.zip  (0.48 GB)
  scene0010_00.zip  (0.81 GB)


## Cell 4 — Extract Zips to Local SSD

Copies zips from Drive to Colab's local SSD and extracts them.
Drive has high per-file latency — local SSD is much faster for the dataloader.
Takes ~5–10 min once per session; already-extracted scenes are skipped.

In [6]:
import os, glob, shutil, subprocess, time

LOCAL_DATA = '/content/scannet_data'
os.makedirs(LOCAL_DATA, exist_ok=True)

zips = sorted(glob.glob(os.path.join(DRIVE_ZIPS, '*.zip')))
print(f'Found {len(zips)} zip(s) to extract\n')

failed = []

for i, zip_drive in enumerate(zips, 1):
    scene = os.path.basename(zip_drive).replace('.zip', '')
    dst = os.path.join(LOCAL_DATA, scene)
    prefix = f'[{i}/{len(zips)}] {scene}'

    if os.path.isdir(dst) and len(glob.glob(os.path.join(dst, 'exported/color/*.jpg'))) > 0:
        n = len(glob.glob(os.path.join(dst, 'exported/color/*.jpg')))
        print(f'{prefix} — already extracted ({n} frames), skipping')
        continue

    size_gb = os.path.getsize(zip_drive) / 1e9
    zip_local = f'/content/{scene}.zip'

    t0 = time.time()
    print(f'{prefix} — copying {size_gb:.2f} GB from Drive...', end=' ', flush=True)
    shutil.copy2(zip_drive, zip_local)
    print(f'done ({time.time()-t0:.0f}s). Extracting...', end=' ', flush=True)

    t1 = time.time()
    result = subprocess.run(['unzip', '-q', zip_local, '-d', LOCAL_DATA])
    os.remove(zip_local)

    if result.returncode != 0:
        print(f'FAILED (unzip error {result.returncode}) — zip may be corrupted on Drive.')
        failed.append(scene)
    else:
        print(f'done ({time.time()-t1:.0f}s).')

total = len(glob.glob(os.path.join(LOCAL_DATA, '*/exported/color/*.jpg')))
print(f'\nAll done. {total} frames in {LOCAL_DATA}')
if failed:
    print(f'\nFailed scenes (re-upload zips to Drive): {failed}')

Found 11 zip(s) to extract

[1/11] scene0000_00 — already extracted (5578 frames), skipping
[2/11] scene0001_00 — already extracted (1544 frames), skipping
[3/11] scene0002_00 — already extracted (5193 frames), skipping
[4/11] scene0003_00 — already extracted (1736 frames), skipping
[5/11] scene0004_00 — already extracted (929 frames), skipping
[6/11] scene0005_00 — already extracted (1159 frames), skipping
[7/11] scene0006_00 — already extracted (2160 frames), skipping
[8/11] scene0007_00 — copying 0.40 GB from Drive... done (12s). Extracting... done (2s).
[9/11] scene0008_00 — copying 0.39 GB from Drive... done (12s). Extracting... done (2s).
[10/11] scene0009_00 — copying 0.48 GB from Drive... done (13s). Extracting... done (3s).
[11/11] scene0010_00 — copying 0.81 GB from Drive... done (27s). Extracting... done (5s).

All done. 23871 frames in /content/scannet_data


## Cell 5 — Download Pretrained Weights
*(Skip on resume — weights persist for the session)*

In [7]:
WEIGHTS_PATH  = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
elif os.path.exists(DRIVE_WEIGHTS):
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
else:
    print('WARNING: weights not found. Upload indoor-lite-LA.ckpt to Drive at:')
    print(' ', DRIVE_WEIGHTS)

Copied weights from Drive


## Cell 6 — Verify GPU & Sanity Check

Runs 50 steps on 5 pairs to confirm the pipeline works end-to-end.  
Loss should decrease. Takes ~1 min on L4.

In [8]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU — no GPU!"}')
print(f'CUDA: {torch.cuda.is_available()}')

# # Sanity check: 5 pairs, 50 steps, reads from Drive (fine for 5 pairs)
# !python train_finetune.py \
#     --overfit \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
#     --batch 4 \
#     --steps 50

# print('Sanity check done — loss should be decreasing!')

GPU: NVIDIA L4
CUDA: True


## Cell 7 — Full Fine-Tuning

**Auto-resume is on** — if Colab disconnects, re-run Cell 1 → 3 → 4 → this cell.  
It will pick up from the last checkpoint in `DRIVE_CKPT_DIR` automatically.

Key settings:
- `--data_dir` reads from **local SSD** (fast)
- `--precision bf16` — ~2x speedup on L4 via native bf16 tensor cores
- `--tau 50.0` — soft epipolar mask, better for multi-scene diversity
- `--save_every 500` — saves to Drive every 500 steps (20 checkpoints total)

In [9]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!python train_finetune.py \
    --data_dir {LOCAL_DATA} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 500 \
    --precision bf16

# --- Resume command (same as above, auto-resume is built in) ---
# !python train_finetune.py \
#     --data_dir {LOCAL_DATA} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --steps 10000 \
#     --tau 50.0 --batch 4 --workers 4 --save_every 500 --precision bf16

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
[Auto-resume] Found checkpoint: /content/drive/MyDrive/final_proj/matchformer_checkpoints_run2/last.ckpt
  Scenes found: 11
    scene0000_00: 5558 pairs
    scene0001_00: 1524 pairs
    scene0002_00: 5173 pairs
    scene0003_00: 1716 pairs
    scene0004_00: 909 pairs
    scene0005_00: 1139 pairs
    scene0006_00: 2140 pairs
    scene0007_00: 1021 pairs
    scene0008_00: 1018 pairs
    scene0009_00: 960 pairs
    scene0010_00: 2493 pairs
  Total pairs: 23651
Dataset: 23651 pairs
2026-03-17 03:00:17.170 | INFO     | model.lightning_loftr:__init__:50 - Load 'model/weights/indoor-lite-LA.ckpt' as pretrained checkpoint
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: Fa

## Cell 8 — Benchmark After Training

In [ ]:
TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
LOCAL_CKPT   = 'model/weights/epipolar-finetuned.ckpt'

shutil.copy(TRAINED_CKPT, LOCAL_CKPT)
print(f'Checkpoint copied to {LOCAL_CKPT}')

!python run_benchmark.py \
    --num_pairs 100 \
    2>&1 | tee benchmark_finetuned_log.txt

shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Results saved to Drive.')